# Model Training with Multiple Algorithms
Trains and compares different ML models to find the best performer

In [ ]:
import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load preprocessed data
X_train = np.load('../Data/processed/X_train.npy')
X_test = np.load('../Data/processed/X_test.npy')
y_train = np.load('../Data/processed/y_train.npy')
y_test = np.load('../Data/processed/y_test.npy')
le = joblib.load('../Models/disease_encoder.pkl')

print(f'Training set: {X_train.shape}')
print(f'Test set: {X_test.shape}')
print(f'Number of classes: {len(np.unique(y_train))}')

In [ ]:
# Define models to try
models = {
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=20, min_samples_split=5,
        class_weight='balanced', n_jobs=-1, random_state=42
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, max_depth=7, learning_rate=0.1, random_state=42
    ),
    'XGBoost': XGBClassifier(
        n_estimators=200, max_depth=7, learning_rate=0.1,
        eval_metric='mlogloss', random_state=42, n_jobs=-1
    ),
    'SVM': SVC(
        C=1.0, kernel='rbf', probability=True, 
        class_weight='balanced', random_state=42
    ),
}

results = []
best_model = None
best_accuracy = 0

for name, model in models.items():
    print(f'\n{"="*50}')
    print(f'Training {name}...')
    print(f'{"="*50}')
    
    # Train
    model.fit(X_train, y_train)
    
    # Predict
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    
    # Evaluate
    train_acc = accuracy_score(y_train, y_pred_train)
    test_acc = accuracy_score(y_test, y_pred_test)
    test_f1 = f1_score(y_test, y_pred_test, average='weighted', zero_division=0)
    
    print(f'Train Accuracy: {train_acc:.4f}')
    print(f'Test Accuracy:  {test_acc:.4f}')
    print(f'Test F1 Score:  {test_f1:.4f}')
    
    results.append({
        'Model': name,
        'Train Accuracy': train_acc,
        'Test Accuracy': test_acc,
        'F1 Score': test_f1,
        'ModelObject': model
    })
    
    if test_acc > best_accuracy:
        best_accuracy = test_acc
        best_model = model
        best_name = name

print(f'\n{"="*50}')
print('SUMMARY')
print(f'{"="*50}')
results_df = pd.DataFrame(results)
print(results_df[['Model', 'Train Accuracy', 'Test Accuracy', 'F1 Score']])
print(f'\n✓ Best Model: {best_name} with {best_accuracy:.4f} accuracy')

In [ ]:
# Detailed evaluation of best model
y_pred_best = best_model.predict(X_test)

print(f'\nDetailed Classification Report:')
print(classification_report(y_test, y_pred_best, 
                          target_names=le.classes_, 
                          zero_division=0))

In [ ]:
# Save best model
joblib.dump(best_model, '../Models/best_model.pkl')
print(f'\n✓ Best model saved to Models/best_model.pkl')